# Softmax Regression: multiclass digit classification

This notebook uses scikit-learn's `LogisticRegression` with `multi_class='multinomial'`, which is Softmax Regression. It predicts one of the ten digit classes (0–9).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X, y = digits.data, digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling is recommended because Logistic Regression uses an iterative solver and L2 regularization.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(multi_class='multinomial', max_iter=2000, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'Log loss: {log_loss(y_test, y_proba):.4f}')
print('\nClassification report:\n', classification_report(y_test, y_pred))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=digits.target_names).plot(cmap='Blues')
plt.title('Softmax Regression: confusion matrix')
plt.show()

## Model, loss, scaling, and evaluation

### Formula notation

- $X \in \mathbb{R}^{n \times d}$: real-valued feature matrix with $n$ examples and $d=64$ pixel features.
- $\mathbf{x}_i$: features for digit image $i$; $y_i \in \{0, \ldots, 9\}$: its true class.
- $K=10$: number of classes; $\mathbf{W} \in \mathbb{R}^{K \times d}$: learned weight matrix; $\mathbf{b} \in \mathbb{R}^{K}$: learned bias vector.
- $z_{ik}$: linear score (logit) for example $i$ and class $k$; $p_{ik}$: predicted probability of that class.

### Softmax model

$$z_{ik} = \mathbf{w}_k^T\mathbf{x}_i + b_k$$

$$p_{ik} = \frac{e^{z_{ik}}}{\sum_{j=1}^{K} e^{z_{ij}}}$$

$$\hat{y}_i = \operatorname*{argmax}_{k} p_{ik}$$

Softmax is used because the ten classes are mutually exclusive. It converts ten linear scores into non-negative probabilities that sum to $1$, then chooses the most probable digit.

### Training objective: multiclass cross-entropy

$$J = -\frac{1}{n}\sum_{i=1}^{n} \log(p_{i,y_i})$$

This loss penalizes the model when it gives low probability to the true digit. **Minimize** it: $0$ is best and there is no fixed worst value.

### Feature scaling

$$x_{ij}^{\mathrm{scaled}} = \frac{x_{ij}-\mu_j}{\sigma_j}$$

Calculate $\mu_j$ and $\sigma_j$ from training data only, then transform test data with those same values. Scaling helps the iterative solver and L2 regularization; fitting the scaler only on training data prevents leakage.

### Evaluation

$$\mathrm{Accuracy} = \frac{1}{m}\sum_{i=1}^{m}\mathbb{1}(\hat{y}_i=y_i)$$

**Maximize accuracy:** its range is $[0,1]$, where $1$ is best. The confusion matrix shows actual classes by row and predicted classes by column. `classification_report` gives precision, recall, and $F_1$ for each digit; maximize all three, where $1$ is best.